### Preprocessing VSG

In [ ]:
import gc
import glob
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

import dask
from dask.diagnostics import ProgressBar
from IPython.display import HTML
from tqdm import tqdm

sys.path.append('..')
from src.utils import parse_ncf_stack_filename
from src.ncf import get_vs_number, export_targeted_ncfs
from src.plots import plot_vsg, animate_vsg

import matplotlib as mpl
mpl.rcParams["animation.html"] = "jshtml"
mpl.rcParams["animation.embed_limit"] = 200.0 

old_fig_dpi = plt.rcParams['figure.dpi']
old_save_dpi = plt.rcParams['savefig.dpi']

#### 1. Read NCFs

In [ ]:
# Choose dataset
pattern = "../data/ncf_stacks_ram/30d/20210928_cc_*_30d_v1.npy"

files_v1 = sorted(glob.glob(pattern), key=get_vs_number)

print("Matched files:", len(files_v1))
print(f"First 3 files: {[os.path.basename(f) for f in files_v1[:3]]}")
print(f"Last 3 files: {[os.path.basename(f) for f in files_v1[-3:]]}")

In [ ]:
# 1. Load ONE file to infer axes 
# We only need one representative NCF to build lag_axis & distance_axis
ncf0_path = files_v1[0]
ncf0 = np.load(ncf0_path)

# Parse metadata (date, vs, window, mode)
date0, vs0, window0, mode0 = parse_ncf_stack_filename(ncf0_path)
print("Example parsed:", date0, vs0, window0, mode0)

In [ ]:
# 2. Define parameters
fs = 250        # Sampling frequency in Hz
dt = 1 / fs     # sampling rate in sec

max_lag = 2.0     # seconds
npts_lag = int(max_lag * fs)

# Array configuration
first_chan = 399
last_chan = 550 # 748 for whole cable
dx = 8.16
channels = np.arange(first_chan, last_chan + 1)

n_rec = len(channels)
lag_axis = np.linspace(-max_lag, +max_lag, ncf0.shape[1])  
distance_axis = (channels - channels[0]) * dx

print(f"NCF shape: {ncf0.shape}, Lag axis: {lag_axis.shape}, Distance axis: {distance_axis.shape}")

#### 2. Spatial-Temporal Swapping/Folding

In [ ]:
output_folder = "../data/ncf_pre_new"

export_targeted_ncfs(
    pattern,
    out_dir=output_folder,
    lag_axis=lag_axis,
    distance_axis=distance_axis,
    target="s1",                 
    dx=dx,
    range_m=500,              
    view_side="both",          
    pos_offset=0.0,            
    vs_start=None,              
    vs_end=None,               
    max_lag=2.0,                  
    taper_alpha=0.1               
)

export_targeted_ncfs(
    pattern,
    out_dir=output_folder,
    lag_axis=lag_axis,
    distance_axis=distance_axis,
    target="s2",                 
    dx=dx,
    range_m=500,              
    view_side="both",          
    pos_offset=0.0,            
    vs_start=None,              
    vs_end=None,               
    max_lag=2.0,                  
    taper_alpha=0.1               
)

#### 3. Display Pre-Processed NCFs

In [ ]:
# Choose dataset
pattern_v1_s1 = "../data/ncf_pre_new/20210928_cc_*_30d_v1_s1.npz"
pattern_v1_s2 = "../data/ncf_pre_new/20210928_cc_*_30d_v1_s2.npz"

files_s1_v1 = sorted(glob.glob(pattern_v1_s1), key=get_vs_number)
files_s2_v1 = sorted(glob.glob(pattern_v1_s2), key=get_vs_number)
print("Matched files (s1):", len(files_s1_v1))
print("Matched files (s2):", len(files_s2_v1))
print("First 3 (s1):", files_s1_v1[:3])
print("First 3 (s2):", files_s2_v1[:3])

In [ ]:
fig, ax = plot_vsg(
    files=files_s1_v1,          
    VS=70,                      
    unit="km",
    pclip=99.0,                
    range_m=500.0,             
    clip_lim=True,
    view_side="both",           
    pos_offset=0.0,            
    show_cbar=True,
    figsize=(6, 6),
    title="Preprocessed VSG (s1)"
)

plt.show()

In [ ]:
try:
    plt.rcParams.update({
        'figure.dpi': 100,        
        'savefig.dpi': 100,
        'axes.titlesize': 14,
    })

    ani_pre = animate_vsg(
    files=files_s1_v1,           
    unit="km",
    pclip=99,
    range_m=500,             
    view_side="both",          
    pos_offset=0.0,           
)
    
    output = HTML(ani_pre.to_jshtml())

finally:
    plt.rcParams.update({
        'figure.dpi': old_fig_dpi,
        'savefig.dpi': old_save_dpi,
    })

display(output)

plt.close('all')
del ani_pre, output
gc.collect()